# Preprocessing

In this notebook, we prepare the selected features for machine learning.

The features are separated into **numerical** and **categorical** variables because they require different preprocessing techniques. Numerical features may need scaling, while categorical features need to be encoded into numerical values.

The target variable `Time_taken(min)` is kept separately as `y`.


In [2]:
import pandas as pd

df = pd.read_csv("../data/feature_engineered_data.csv")

In [3]:
y = df["Time_taken(min)"]

X = df.drop(columns=["Time_taken(min)", "Order_Date"])

In [4]:
X = X.drop(columns=[
    "Restaurant_latitude",
    "Restaurant_longitude",
    "Delivery_location_latitude",
    "Delivery_location_longitude",
    "is_weekend"
])

In [5]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (41522, 11)
y shape: (41522,)


## Separate Numerical and Categorical Features

We separate the features into two groups:

* **Numerical features:** contain numbers.
* **Categorical features:** contain categories or text.

This allows us to apply the appropriate preprocessing technique to each group.


In [7]:
numerical_features = X.select_dtypes(
    include=["int64","float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object","string"]
).columns.to_list()

In [8]:
print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

Numerical features: ['Delivery_person_Age', 'Delivery_person_Ratings', 'multiple_deliveries', 'distance', 'order_hour', 'pickup_hour', 'month']
Categorical features: ['Weatherconditions', 'Type_of_vehicle', 'Road_traffic_density', 'day_of_week']


## Inspect Categorical Features

Before encoding, we check the unique values of each categorical feature.

This helps us understand the categories that will be converted into numerical values.


In [9]:
for col in categorical_features:
    print(f"\n{col}:")
    print(X[col].unique())


Weatherconditions:
<ArrowStringArray>
['Sunny', 'Stormy', 'Sandstorms', 'Cloudy', 'Fog', 'Windy']
Length: 6, dtype: str

Type_of_vehicle:
<ArrowStringArray>
['motorcycle ', 'scooter ', 'electric_scooter ', 'bicycle ']
Length: 4, dtype: str

Road_traffic_density:
<ArrowStringArray>
['High ', 'Jam ', 'Low ', 'Medium ', 'Unknown']
Length: 5, dtype: str

day_of_week:
<ArrowStringArray>
['Saturday', 'Friday', 'Tuesday', 'Monday', 'Sunday', 'Wednesday', 'Thursday']
Length: 7, dtype: str


In [10]:
for col in categorical_features:
    X[col] = X[col].str.strip()

X["Road_traffic_density"] = X["Road_traffic_density"].replace(
    "Unknown",
    X["Road_traffic_density"].mode()[0]
)

In [11]:
for col in categorical_features:
    print(f"\n{col}:")
    print(X[col].unique())


Weatherconditions:
<ArrowStringArray>
['Sunny', 'Stormy', 'Sandstorms', 'Cloudy', 'Fog', 'Windy']
Length: 6, dtype: str

Type_of_vehicle:
<ArrowStringArray>
['motorcycle', 'scooter', 'electric_scooter', 'bicycle']
Length: 4, dtype: str

Road_traffic_density:
<ArrowStringArray>
['High', 'Jam', 'Low', 'Medium']
Length: 4, dtype: str

day_of_week:
<ArrowStringArray>
['Saturday', 'Friday', 'Tuesday', 'Monday', 'Sunday', 'Wednesday', 'Thursday']
Length: 7, dtype: str


## One-Hot Encoding

Machine learning models generally work with numerical values, so categorical features must be converted into numbers.

We use **One-Hot Encoding** to create a binary column for each category.


In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore", #avoids errors if a new category appears lateeer
    sparse_output=False # retuuurns a normal NumPy array
)

## Apply One-Hot Encoding

We apply the encoder to the categorical features.

Each category becomes a separate binary column containing `0` or `1`.


In [17]:
X_categorical = encoder.fit_transform(
    X[categorical_features]
)

## Check Encoded Features

We check the shape of the encoded data and verify that the transformation was successful.


In [18]:
print("Encoded categorical shape:", X_categorical.shape)

Encoded categorical shape: (41522, 21)


## Create Encoded DataFrame

The encoder returns a NumPy array. We convert it into a DataFrame and keep the generated feature names.


In [19]:
encoded_df = pd.DataFrame(
    X_categorical,
    columns=encoder.get_feature_names_out(categorical_features),
    index=X.index
)

In [24]:
print(encoded_df)

       Weatherconditions_Cloudy  Weatherconditions_Fog  \
0                           0.0                    0.0   
1                           0.0                    0.0   
2                           0.0                    0.0   
3                           0.0                    0.0   
4                           1.0                    0.0   
...                         ...                    ...   
41517                       0.0                    0.0   
41518                       0.0                    0.0   
41519                       1.0                    0.0   
41520                       1.0                    0.0   
41521                       0.0                    1.0   

       Weatherconditions_Sandstorms  Weatherconditions_Stormy  \
0                               0.0                       0.0   
1                               0.0                       1.0   
2                               1.0                       0.0   
3                               0.0        

## Select Numerical Features

We keep the 7 numerical features separately so they can be combined with the encoded categorical features.


In [22]:
X_numerical = X[numerical_features]

In [23]:
print("Numerical shape:", X_numerical.shape)
print("Encoded categorical shape:", encoded_df.shape)

Numerical shape: (41522, 7)
Encoded categorical shape: (41522, 21)


## Combine Preprocessed Features

The numerical features and the encoded categorical features are currently stored separately.

We combine them into one DataFrame so that the model can use all features together.


In [25]:
X_preprocessed = pd.concat(
    [X_numerical, encoded_df],
    axis=1
)

In [26]:
print("Preprocessed shape:", X_preprocessed.shape)

Preprocessed shape: (41522, 28)


## Numerical Feature Scaling

Numerical features can have different ranges and scales. Scaling puts them on a comparable scale so that features with larger numerical values do not unnecessarily dominate the model.

The encoded categorical features are already represented as `0` and `1`, so we do not scale them.



## Train/Test Split

We split the dataset into a training set and a test set.

* **Training set:** used to train the model.
* **Test set:** kept unseen and used later to evaluate the model.

We use 80% of the data for training and 20% for testing.


In [28]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)



## Verify the Split

We check the shapes of the training and test sets to make sure the data was split correctly.


In [29]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (33217, 11)
X_test shape: (8305, 11)
y_train shape: (33217,)
y_test shape: (8305,)


## Separate Numerical and Categorical Features

Numerical and categorical features require different preprocessing.

* Numerical features will be scaled.
* Categorical features will be encoded.


In [30]:
numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "string"]
).columns.tolist()

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['Delivery_person_Age', 'Delivery_person_Ratings', 'multiple_deliveries', 'distance', 'order_hour', 'pickup_hour', 'month']

Categorical features:
['Weatherconditions', 'Type_of_vehicle', 'Road_traffic_density', 'day_of_week']


## Standardization of Numerical Features

Numerical features have different ranges, so we standardize them using `StandardScaler`.

The scaler is fitted only on the training data to avoid data leakage.


In [31]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_numerical = scaler.fit_transform(
    X_train[numerical_features]
)

X_test_numerical = scaler.transform(
    X_test[numerical_features]
)

In [32]:
print("Train numerical shape:", X_train_numerical.shape)
print("Test numerical shape:", X_test_numerical.shape)

Train numerical shape: (33217, 7)
Test numerical shape: (8305, 7)


## One-Hot Encoding

Categorical features cannot be used directly by most machine learning models.

We use One-Hot Encoding to convert each category into a binary feature.


In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_categorical = encoder.fit_transform(
    X_train[categorical_features]
)

X_test_categorical = encoder.transform(
    X_test[categorical_features]
)

In [34]:
print("Train categorical shape:", X_train_categorical.shape)
print("Test categorical shape:", X_test_categorical.shape)

Train categorical shape: (33217, 21)
Test categorical shape: (8305, 21)


In [37]:
scaled_train_df = pd.DataFrame(
    X_train_numerical,
    columns=numerical_features,
    index=X_train.index
)

scaled_test_df = pd.DataFrame(
    X_test_numerical,
    columns=numerical_features,
    index=X_test.index
)

## Convert Encoded Features to DataFrames

The encoded categorical features are stored as NumPy arrays. We convert them into DataFrames and keep the generated feature names.


In [35]:
encoded_train_df = pd.DataFrame(
    X_train_categorical,
    columns=encoder.get_feature_names_out(categorical_features),
    index=X_train.index
)

encoded_test_df = pd.DataFrame(
    X_test_categorical,
    columns=encoder.get_feature_names_out(categorical_features),
    index=X_test.index
)

## Combine Preprocessed Features

The numerical and categorical features are now preprocessed separately.

We combine them into a single feature matrix that can be used by the machine learning model.


In [38]:
X_train_preprocessed = pd.concat(
    [scaled_train_df, encoded_train_df],
    axis=1
)

X_test_preprocessed = pd.concat(
    [scaled_test_df, encoded_test_df],
    axis=1
)

In [39]:
print("X_train_preprocessed shape:", X_train_preprocessed.shape)
print("X_test_preprocessed shape:", X_test_preprocessed.shape)

X_train_preprocessed shape: (33217, 28)
X_test_preprocessed shape: (8305, 28)


## Verify Preprocessed Data

We verify the shape and structure of the final training and test features before using them to train the model.


In [40]:
print("X_train_preprocessed shape:", X_train_preprocessed.shape)
print("X_test_preprocessed shape:", X_test_preprocessed.shape)

print("\nMissing values in train:",
      X_train_preprocessed.isnull().sum().sum())

print("Missing values in test:",
      X_test_preprocessed.isnull().sum().sum())

X_train_preprocessed shape: (33217, 28)
X_test_preprocessed shape: (8305, 28)

Missing values in train: 0
Missing values in test: 0
